 # Part 1: AlphaCache Generation (CPU-Optimized)

 This notebook loads raw market and macro data, sets up the Universe Screener,

 and builds the multi-day lookback Feature Cube (`AlphaCache`).

 The resulting feature cube is saved to disk for the training notebook.

In [1]:
import time

start_time = time.time()

In [2]:
# CELL 1: Setup, Environment Switch, and Path Configuration

import os
import sys
from pathlib import Path

os.environ["CACHE_LOOKBACK"] = "189"
os.environ["CACHE_START_DATE"] = "2020-01-01"

# ============================================================================
# --- ENVIRONMENT SWITCH ---
# Set to True if running in Google Colab.
# Set to False if running locally in Windows PC VSCode.
# ============================================================================
RUNNING_IN_COLAB = True

if RUNNING_IN_COLAB:
    from google.colab import drive

    # 1. Mount Google Drive
    drive.mount("/content/drive")

    # 2. Define codebase paths
    project_path = Path(
        "/content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks"
    )
    codebase_path = project_path / "notebooks_RLVR_v2"
else:
    # 1. Local Windows VSCode Paths
    project_path = Path("C:/Users/ping/Files_win10/python/py311/stocks")
    codebase_path = project_path / "notebooks_RLVR_v2"

data_path = project_path / "data" / "processed"
data_path.mkdir(parents=True, exist_ok=True)

output_path = codebase_path / "output"
output_path.mkdir(parents=True, exist_ok=True)

# 3. Change the Current Working Directory
os.chdir(codebase_path)

# 4. Add to system path so absolute imports work
if str(codebase_path) not in sys.path:
    sys.path.append(str(codebase_path))

# Verify environment visibility
print("\n--- Environment Check ---")
print(f"Current Directory: {os.getcwd()}")
visible_files = os.listdir()
print(f"Files visible here: {visible_files}")

if "core" in visible_files and "rl_discovery" in visible_files:
    print("\n[OK] SUCCESS: Python can see your architecture! Imports will now work.")
    print(f"project_path: {project_path}")
    print(f"data_path: {data_path}")
    print(f"codebase_path: {codebase_path}")
    print(f"output_path: {output_path}")
else:
    print("\n[ERROR] FAILURE: Python cannot see 'core' or 'rl_discovery'.")
    print("Please check your directory configuration and folder structure.")

Mounted at /content/drive

--- Environment Check ---
Current Directory: /content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks/notebooks_RLVR_v2
Files visible here: ['core', 'strategy', 'tests', 'output', 'data_pipeline', 'rl_discovery', 'walk_forward', 'metric_definitions.md', 'main.py', '__pycache__', 'nb2txt.py', 'data', 'archive', '0_RLVR_data_process.ipynb', 'main.ipynb', 'features_df_out.xlsx', 'verify_UI_n_features_calc_v4.ipynb', '_trash.md', '_trash.ipynb', 'run_pipeline.py', 'fed_data_yield_spread.ipynb', 'nb2txt.ipynb', 'verify_alphacache.ipynb', 'run_pipeline.ipynb', 'TODO.md', 'RLVR_Part2_Training_v2.ipynb', 'RLVR_Part1_AlphaCache_v2.ipynb']

[OK] SUCCESS: Python can see your architecture! Imports will now work.
project_path: /content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks
data_path: /content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks/data/processed
codebase_path: /content/drive/Othercomputers/My Computer/F

In [3]:
# CELL 2: Load Raw Data into RAM
import pandas as pd

print("Loading DataFrames into RAM. Please wait...")
df_ohlcv = pd.read_parquet(project_path / "data" / "df_OHLCV_stocks_etfs.parquet")

macro_df = pd.read_parquet(data_path / "macro_df.parquet")
features_df = pd.read_parquet(data_path / "features_df.parquet")

print("Data loaded successfully.")

Loading DataFrames into RAM. Please wait...
Data loaded successfully.


In [4]:
# CELL 3: Imports
from core.settings import TradingConfig, CacheConfig  # Imported shared configuration
from data_pipeline.screener import UniverseScreener
from data_pipeline.cache import AlphaCache

In [5]:
# CELL 4: Data Pipeline Preparation & Alpha Cache Generation
print("\n--- [TASK 1] Initializing Universe and AlphaCache ---")

# Setup Config & Extract Wide Close Prices
config = TradingConfig()

# Convert MultiIndex OHLCV to Wide Format (Dates x Tickers)
df_close = df_ohlcv["Adj Close"].unstack(level=0)

# Ensure clean calendar
trading_calendar = pd.DatetimeIndex(df_close.dropna(how="all").index.sort_values())

# 1. Initialize Screener (The Gatekeeper)
screener = UniverseScreener(
    df_close=df_close,
    features_df=features_df,
    macro_df=macro_df,
    trading_calendar=trading_calendar,
    config=config,
)

# 1. Instantiate Cache using central Lookback constant
cache = AlphaCache(screener=screener, config=config, lookbacks=[CacheConfig.LOOKBACK])

# 2. Build starting from central Start Date constant
train_start = CacheConfig.START_DATE
cache.build(start_date=train_start)

# 3. Save to the centrally managed dynamic filename
cache_file_path = output_path / CacheConfig.get_filename()
cache.feature_cube.to_parquet(cache_file_path)
print(f"\n[SAVED] AlphaCache successfully written to {cache_file_path}")


--- [TASK 1] Initializing Universe and AlphaCache ---
[INFO] Building AlphaCache for 1620 days (Starting 2020-01-01)...
  Processed 0/1620 days...
  Processed 20/1620 days...
  Processed 40/1620 days...
  Processed 60/1620 days...
  Processed 80/1620 days...
  Processed 100/1620 days...
  Processed 120/1620 days...
  Processed 140/1620 days...
  Processed 160/1620 days...
  Processed 180/1620 days...
  Processed 200/1620 days...
  Processed 220/1620 days...
  Processed 240/1620 days...
  Processed 260/1620 days...
  Processed 280/1620 days...
  Processed 300/1620 days...
  Processed 320/1620 days...
  Processed 340/1620 days...
  Processed 360/1620 days...
  Processed 380/1620 days...
  Processed 400/1620 days...
  Processed 420/1620 days...
  Processed 440/1620 days...
  Processed 460/1620 days...
  Processed 480/1620 days...
  Processed 500/1620 days...
  Processed 520/1620 days...
  Processed 540/1620 days...
  Processed 560/1620 days...
  Processed 580/1620 days...
  Processed 600

In [6]:
import pandas as pd

# 2. Re-construct the dynamic cache file path
cache_file_path = output_path / CacheConfig.get_filename()

# 3. Load the cache parquet file back into a Pandas DataFrame
if cache_file_path.exists():
    feature_cube = pd.read_parquet(cache_file_path)
    print(f"[OK] Successfully loaded AlphaCache!")
    print(f"Path:  {cache_file_path}")
    print(f"Shape: {feature_cube.shape}")

    # Show a quick preview of your features
    display(feature_cube.head())
else:
    print(f"[ERROR] Cache file not found at: {cache_file_path}")

[OK] Successfully loaded AlphaCache!
Path:  /content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks/notebooks_RLVR_v2/output/alpha_cache_189d_2020.parquet
Shape: (1420001, 11)


189d_Log Price Gain  189d_Sharpe (TRP)  \
Date       Ticker                                           
2020-01-02 A                  0.054236           0.020174   
           AA                -0.292223          -0.032557   
           AAL               -0.137770          -0.016347   
           AAPL               0.440790           0.122948   
           ABBV               0.117776           0.036156   

                   189d_Momentum (21d)  189d_Info Ratio (63d)  \
Date       Ticker                                               
2020-01-02 A                  0.071955               0.089283   
           AA                 0.053616              -0.003946   
           AAL                0.035969               0.023447   
           AAPL               0.137000               0.367420   
           ABBV               0.028955               0.181767   

                   189d_Oversold (-RSI)  189d_Dip Buyer (-dd_21)  \
Date       Ticker                                                  
2020-01-02 A                 -71.708262                -0.000000   
           AA                -55.138168                 0.007417   
           AAL               -55.177878                 0.019550   
           AAPL              -84.608288                -0.000000   
           ABBV              -59.729697                 0.007756   

                   189d_Range Position (20d)  189d_Return Autocorr (15d)  \
Date       Ticker                                                          
2020-01-02 A                        0.936898                   -0.237952   
           AA                       0.769566                   -0.603651   
           AAL                      0.769978                   -0.176547   
           AAPL                     0.993738                   -0.187326   
           ABBV                     0.584322                   -0.193186   

                   189d_Low Volatility (-ATRP)  189d_OBV Divergence (5d)  \
Date       Ticker                                                          
2020-01-02 A                         -0.020804                  0.080578   
           AA                        -0.039522                 -0.114050   
           AAL                       -0.030902                  0.192644   
           AAPL                      -0.019641                  0.453090   
           ABBV                      -0.021574                 -0.022538   

                   189d_Convexity  
Date       Ticker                  
2020-01-02 A             0.001955  
           AA            0.000702  
           AAL           0.006405  
           AAPL          0.001242  
           ABBV          0.003230

In [7]:
feature_cube.info()

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 1420001 entries, (Timestamp('2020-01-02 00:00:00'), 'A') to (Timestamp('2026-06-04 00:00:00'), 'ZTS')
Data columns (total 11 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   189d_Log Price Gain          1420001 non-null  float64
 1   189d_Sharpe (TRP)            1420001 non-null  float64
 2   189d_Momentum (21d)          1420001 non-null  float64
 3   189d_Info Ratio (63d)        1419938 non-null  float64
 4   189d_Oversold (-RSI)         1420001 non-null  float64
 5   189d_Dip Buyer (-dd_21)      1420001 non-null  float64
 6   189d_Range Position (20d)    1420001 non-null  float64
 7   189d_Return Autocorr (15d)   1420001 non-null  float64
 8   189d_Low Volatility (-ATRP)  1420001 non-null  float64
 9   189d_OBV Divergence (5d)     1420001 non-null  float64
 10  189d_Convexity               1420001 non-null  float64
dtypes: float64(11)
memory usage: 1

In [8]:
end_time = time.time()
print(f"Script took: {end_time - start_time:.2f} seconds")

Script took: 3811.12 seconds
